# Experiment 3: Scaling Comparison
In this notebook, we evaluate how different scaling techniques for numeric features affect model performance.

### Why Scale?
- **Distance-based models**: KNN and SVM rely on calculating distances. If one feature has a much larger range (e.g., 0-1000) than another (e.g., 0-1), it will dominate the distance metric.
- **Gradient-based models**: Logistic Regression and Neural Networks converge faster when features are on similar scales.

### Scalers to Test:
1. **StandardScaler**: Centers data around 0 with a standard deviation of 1.
2. **MinMaxScaler**: Squashes data into a specific range, usually [0, 1]. Sensitive to outliers.
3. **RobustScaler**: Uses median and interquartile range (IQR), making it more robust to outliers.
4. **No Scaler**: Baseline to see the impact of scaling.

In [3]:
import pandas as pd
import numpy as np
import mlflow
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, classification_report
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler
from scipy.sparse import hstack



In [4]:
mlflow.set_tracking_uri("http://localhost:5000")
mlflow.set_experiment("Scaling Techniques Comparison - 2")

2026/04/14 11:38:46 INFO mlflow.tracking.fluent: Experiment with name 'Scaling Techniques Comparison - 2' does not exist. Creating a new experiment.


<Experiment: artifact_location='mlflow-artifacts:/14', creation_time=1776146926373, experiment_id='14', last_update_time=1776146926373, lifecycle_stage='active', name='Scaling Techniques Comparison - 2', tags={}, workspace='default'>

In [5]:
# Load and preprocess
data = pd.read_csv(r'../data/processed/dataset.csv')
data = data.dropna(subset=['text_processed', 'sentiment']).drop_duplicates()

X_text = data['text_processed']
X_numeric = data[['char_count', 'word_count', 'avg_word_len']].values
y = data['sentiment']

X_text_train, X_text_test, X_num_train, X_num_test, y_train, y_test = train_test_split(
    X_text, X_numeric, y, test_size=0.2, random_state=42, stratify=y
)

# Text representation is fixed for this experiment
tfidf = TfidfVectorizer(max_features=2500, ngram_range=(1,1))
X_text_train_vec = tfidf.fit_transform(X_text_train)
X_text_test_vec = tfidf.transform(X_text_test)

In [6]:
scalers = {
    "NoScaling": None,
    "StandardScaler": StandardScaler(),
    "MinMaxScaler": MinMaxScaler(),
    "RobustScaler": RobustScaler()
}

for name, scaler in scalers.items():
    with mlflow.start_run(run_name=f"Scaling_{name}"):
        if scaler:
            X_num_train_scaled = scaler.fit_transform(X_num_train)
            X_num_test_scaled = scaler.transform(X_num_test)
        else:
            X_num_train_scaled = X_num_train
            X_num_test_scaled = X_num_test
            
        X_train_final = hstack([X_text_train_vec, X_num_train_scaled])
        X_test_final = hstack([X_text_test_vec, X_num_test_scaled])
        
        # Train
        model = LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced')
        model.fit(X_train_final, y_train) # type: ignore
        y_pred = model.predict(X_test_final)
        
        # ── Log Parameters ────────────────────────────────────────────────────────
        mlflow.log_params({
                   # Split
            "test_size"             : 0.2,
            "stratify"              : True,
            "random_state"          : 42,

            # Feature Representation
            "representation"        : "TFIDF_2500",
            "vectorizer_type"       : "TfidfVectorizer",
            "tfidf_max_features"    : 2500,
            "ngram_range"           : "(1,1)",

            # Scaling
            "scaler"                : name,
            "scaled_features"       : "char_count, word_count, avg_word_len",

            # Model
            "model"                 : "LogisticRegression",
            "max_iter"              : 1000,
            "solver"                : "lbfgs",
            "random_state"          : 42,
            "class_weight"          : "balanced",

            # Features
            "num_custom_features"   : 3,
            "text_features_count"   : X_text_train_vec.shape[1],
        })

        # Metrics Calculation
        report_dict:dict = classification_report(y_test, y_pred, output_dict=True) # type: ignore
        
        metrics = {
            "accuracy"         : accuracy_score(y_test, y_pred),
            "f1_weighted"      : report_dict['weighted avg']['f1-score'],
            "f1_macro"         : report_dict['macro avg']['f1-score'],
            "precision_weighted": report_dict['weighted avg']['precision'],
            "precision_macro"  : report_dict['macro avg']['precision'],
            "recall_weighted"  : report_dict['weighted avg']['recall'],
            "recall_macro"     : report_dict['macro avg']['recall'],
        }
        
        # Log Per-Class Metrics
        for label in model.classes_:
            metrics[f"f1_class_{label}"] = report_dict[str(label)]['f1-score']
            metrics[f"precision_class_{label}"] = report_dict[str(label)]['precision']
            metrics[f"recall_class_{label}"] = report_dict[str(label)]['recall']
            
        mlflow.log_metrics(metrics)
        
        print(f"Scaler: {name} -> F1 (Weighted): {metrics['f1_weighted']:.4f}")


c:\data-science-projects\major-projects\youtube_comment_analysis\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Scaler: NoScaling -> F1 (Weighted): 0.8868
🏃 View run Scaling_NoScaling at: http://localhost:5000/#/experiments/14/runs/b651f73af5c341eb9e20e51927a6639a
🧪 View experiment at: http://localhost:5000/#/experiments/14
Scaler: StandardScaler -> F1 (Weighted): 0.8877
🏃 View run Scaling_StandardScaler at: http://localhost:5000/#/experiments/14/runs/9fbe491d05c24214b6a14a4c15573dbd
🧪 View experiment at: http://localhost:5000/#/experiments/14
Scaler: MinMaxScaler -> F1 (Weighted): 0.8877
🏃 View run Scaling_MinMaxScaler at: http://localhost:5000/#/experiments/14/runs/e0eb4e0077ad46d0acc3fb3a43805f2a
🧪 View experiment at: http://localhost:5000/#/experiments/14
Scaler: RobustScaler -> F1 (Weighted): 0.8897
🏃 View run Scaling_RobustScaler at: http://localhost:5000/#/experiments/14/runs/b814ebbe4430462384660195cae89333
🧪 View experiment at: http://localhost:5000/#/experiments/14


#### Conclusion
Robust scaling is doing good on custom features although tree based models has lesser no/ significane of scaling features. 

for models dependent of distances and gradient algos makes difference